In [153]:
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import adfuller
from pathlib import Path

In [154]:
DATASET_PATH = Path('..\\Dataset\\PowerLoad_Dataset.csv')

df = pd.read_csv(DATASET_PATH, index_col='Timestamp')

In [155]:
df.head(10)

,Power_Load_kW,Temperature_C,Humidity_%,WindSpeed_mps,Precipitation_mm,DayOfWeek,HolidayFlag,Daily_PostDispatch_Load,Weekly_PreDispatch_Projection
Timestamp,,,,,,,,,
2018-01-01 01:00:00,493.09,20.47,66.90,10.35,0.51,1,0,508.96,519.14
2018-01-01 04:00:00,488.29,27.66,58.72,7.86,0.02,1,0,522.95,533.41
2018-01-01 07:00:00,538.37,15.35,36.77,4.67,0.05,1,0,527.55,538.10
2018-01-01 13:00:00,404.34,22.89,83.71,11.05,0.22,1,0,506.71,516.85
2018-01-02 14:00:00,433.59,20.38,43.12,0.51,0.18,2,0,481.05,498.31
2018-01-02 16:00:00,536.92,23.20,30.63,8.09,0.05,2,0,486.28,500.04
2018-01-03 01:00:00,411.85,14.81,39.98,0.28,0.10,3,0,484.80,498.50
2018-01-03 13:00:00,490.72,19.23,69.90,1.16,0.05,3,0,495.15,501.82
2018-01-03 14:00:00,444.68,23.90,41.42,5.62,0.32,3,0,495.61,501.05


In [156]:
df.info()

<class 'pandas.DataFrame'>
Index: 10000 entries, 2018-01-01 01:00:00 to 2023-06-30 16:00:00
Data columns (total 9 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Power_Load_kW                  10000 non-null  float64
 1   Temperature_C                  10000 non-null  float64
 2   Humidity_%                     10000 non-null  float64
 3   WindSpeed_mps                  10000 non-null  float64
 4   Precipitation_mm               10000 non-null  float64
 5   DayOfWeek                      10000 non-null  int64  
 6   HolidayFlag                    10000 non-null  int64  
 7   Daily_PostDispatch_Load        10000 non-null  float64
 8   Weekly_PreDispatch_Projection  10000 non-null  float64
dtypes: float64(7), int64(2)
memory usage: 781.2+ KB


In [157]:
print(df.shape)
print(df.columns)
print(df.dtypes)
print(df.isna().sum())

(10000, 9)
Index(['Power_Load_kW', 'Temperature_C', 'Humidity_%', 'WindSpeed_mps',
       'Precipitation_mm', 'DayOfWeek', 'HolidayFlag',
       'Daily_PostDispatch_Load', 'Weekly_PreDispatch_Projection'],
      dtype='str')
Power_Load_kW                    float64
Temperature_C                    float64
Humidity_%                       float64
WindSpeed_mps                    float64
Precipitation_mm                 float64
DayOfWeek                          int64
HolidayFlag                        int64
Daily_PostDispatch_Load          float64
Weekly_PreDispatch_Projection    float64
dtype: object
Power_Load_kW                    0
Temperature_C                    0
Humidity_%                       0
WindSpeed_mps                    0
Precipitation_mm                 0
DayOfWeek                        0
HolidayFlag                      0
Daily_PostDispatch_Load          0
Weekly_PreDispatch_Projection    0
dtype: int64


In [158]:
def split_dataset(df, train_size=0.7, validation_size=0.15, test_size=0.15) -> tuple:
    train_end = int(train_size * len(df))
    validation_end = int((train_size + validation_size) * len(df))
    
    train = df[:train_end]
    validation = df[train_end:validation_end]
    test = df[validation_end:]
    
    return train, validation, test

In [159]:
train_df, validation_df, test_df = split_dataset(df)

In [160]:
print(f"Original dataset shape: {df.shape}")
print(f"Train set shape: {train_df.shape}")
print(f"Validation set shape: {validation_df.shape}")
print(f"Test set shape: {test_df.shape}")
print(df.isna().sum())

Original dataset shape: (10000, 9)
Train set shape: (7000, 9)
Validation set shape: (1500, 9)
Test set shape: (1500, 9)
Power_Load_kW                    0
Temperature_C                    0
Humidity_%                       0
WindSpeed_mps                    0
Precipitation_mm                 0
DayOfWeek                        0
HolidayFlag                      0
Daily_PostDispatch_Load          0
Weekly_PreDispatch_Projection    0
dtype: int64


In [161]:
df = df[['Power_Load_kW']]
df['timestamp'] = pd.to_datetime(df.index)
df = df.sort_values("timestamp").reset_index(drop=True)

In [162]:
df.head()

,Power_Load_kW,timestamp
0,493.09,2018-01-01 01:00:00
1,488.29,2018-01-01 04:00:00
2,538.37,2018-01-01 07:00:00
3,404.34,2018-01-01 13:00:00
4,433.59,2018-01-02 14:00:00


In [163]:
time_diff = df["timestamp"].diff()

print(time_diff.value_counts().head(10))

timestamp
0 days 01:00:00    2118
0 days 02:00:00    1600
0 days 03:00:00    1305
0 days 04:00:00    1037
0 days 05:00:00     815
0 days 06:00:00     637
0 days 07:00:00     470
0 days 08:00:00     445
0 days 09:00:00     324
0 days 10:00:00     237
Name: count, dtype: int64


In [164]:
print(df[["timestamp", "Power_Load_kW"]].head(50).to_string(index=False))

          timestamp  Power_Load_kW
2018-01-01 01:00:00         493.09
2018-01-01 04:00:00         488.29
2018-01-01 07:00:00         538.37
2018-01-01 13:00:00         404.34
2018-01-02 14:00:00         433.59
2018-01-02 16:00:00         536.92
2018-01-03 01:00:00         411.85
2018-01-03 13:00:00         490.72
2018-01-03 14:00:00         444.68
2018-01-03 22:00:00         518.07
2018-01-03 23:00:00         576.90
2018-01-04 03:00:00         541.10
2018-01-04 14:00:00         545.77
2018-01-05 00:00:00         514.81
2018-01-05 02:00:00         500.26
2018-01-05 07:00:00         459.89
2018-01-05 12:00:00         512.88
2018-01-05 13:00:00         496.28
2018-01-05 14:00:00         404.06
2018-01-05 20:00:00         498.26
2018-01-06 08:00:00         504.98
2018-01-07 00:00:00         512.99
2018-01-07 03:00:00         433.98
2018-01-07 06:00:00         512.52
2018-01-07 13:00:00         523.69
2018-01-07 23:00:00         594.84
2018-01-08 03:00:00         459.21
2018-01-08 05:00:00 

In [165]:
print(df[["timestamp", "Power_Load_kW"]].tail(50).to_string(index=False))

          timestamp  Power_Load_kW
2023-06-20 02:00:00         432.30
2023-06-20 06:00:00         365.51
2023-06-20 07:00:00         538.37
2023-06-20 12:00:00         458.40
2023-06-20 15:00:00         477.85
2023-06-20 17:00:00         409.78
2023-06-20 21:00:00         462.32
2023-06-21 10:00:00         533.28
2023-06-21 15:00:00         589.60
2023-06-21 16:00:00         477.73
2023-06-21 17:00:00         529.60
2023-06-21 19:00:00         545.63
2023-06-21 23:00:00         482.46
2023-06-22 10:00:00         508.12
2023-06-22 14:00:00         572.16
2023-06-22 22:00:00         434.54
2023-06-23 03:00:00         457.99
2023-06-23 05:00:00         544.59
2023-06-24 02:00:00         486.53
2023-06-24 13:00:00         564.33
2023-06-24 17:00:00         522.95
2023-06-24 21:00:00         426.35
2023-06-25 17:00:00         475.98
2023-06-25 18:00:00         500.96
2023-06-25 19:00:00         521.53
2023-06-26 00:00:00         512.14
2023-06-26 11:00:00         518.15
2023-06-26 14:00:00 

In [166]:
time_diff = df["timestamp"].diff()

print(
    df.loc[
        time_diff > pd.Timedelta(hours=1),
        ["timestamp", "Power_Load_kW"]
    ].head(30).to_string(index=False)
)

          timestamp  Power_Load_kW
2018-01-01 04:00:00         488.29
2018-01-01 07:00:00         538.37
2018-01-01 13:00:00         404.34
2018-01-02 14:00:00         433.59
2018-01-02 16:00:00         536.92
2018-01-03 01:00:00         411.85
2018-01-03 13:00:00         490.72
2018-01-03 22:00:00         518.07
2018-01-04 03:00:00         541.10
2018-01-04 14:00:00         545.77
2018-01-05 00:00:00         514.81
2018-01-05 02:00:00         500.26
2018-01-05 07:00:00         459.89
2018-01-05 12:00:00         512.88
2018-01-05 20:00:00         498.26
2018-01-06 08:00:00         504.98
2018-01-07 00:00:00         512.99
2018-01-07 03:00:00         433.98
2018-01-07 06:00:00         512.52
2018-01-07 13:00:00         523.69
2018-01-07 23:00:00         594.84
2018-01-08 03:00:00         459.21
2018-01-08 05:00:00         517.06
2018-01-08 08:00:00         500.65
2018-01-08 12:00:00         531.28
2018-01-08 14:00:00         446.46
2018-01-08 18:00:00         523.66
2018-01-09 02:00:00 

In [167]:
print("Number of rows:", len(df))
print("Start:", df["timestamp"].min())
print("End:", df["timestamp"].max())

Number of rows: 10000
Start: 2018-01-01 01:00:00
End: 2023-06-30 16:00:00


In [168]:
print(
    "Continuous hourly intervals:",
    (time_diff == pd.Timedelta(hours=1)).sum()
)

Continuous hourly intervals: 2118


In [169]:
print(
    df["timestamp"].dt.date.nunique()
)

2001


In [170]:
print(
    df["timestamp"].dt.year.value_counts().sort_index()
)

timestamp
2018    1843
2019    1753
2020    1814
2021    1881
2022    1798
2023     911
Name: count, dtype: int64
